# Gold — fact_horizon_performance

`silver.monthly_performance` + `gold.dim_ticker` → **`gold.fact_horizon_performance`**.

**Grain: one row per (ticker, horizon).** Five horizons — 15, 10, 5, 3 and 1 year. **This is
the table that answers the research question.**

Built entirely from CTEs, with three window functions doing work that would otherwise need
repeated aggregation:

| window | what it does |
|---|---|
| `SUM(LN(1+r)) OVER (ORDER BY month_key)` | SPY compounded at every month, in one pass. Adding logarithms is multiplying, and SQL has no running product |
| the same frame ending `1 PRECEDING` | the curve shifted one month, so any span becomes one point divided by another |
| `RANK() OVER (PARTITION BY horizon_years ORDER BY ...)` | three leaderboards — return, income, risk-adjusted — in the same pass |

Expected: **445 rows**.

In [0]:
CREATE TABLE IF NOT EXISTS `index-vs-trust-pipeline`.gold.fact_horizon_performance (
  horizon_key                  STRING  COMMENT 'MD5(ticker|horizon_years)',
  ticker_key                   STRING  COMMENT 'The current dim_ticker version: this is an as-of-today summary',
  ticker                       STRING,
  month_key                    INT     COMMENT 'The as-of month, so dim_date joins this fact directly too',
  horizon_years                INT     COMMENT '15, 10, 5, 3 or 1',
  start_month_key              INT     COMMENT 'The span actually used: visible, not implied',
  end_month_key                INT,
  months_used                  INT,
  total_return                 DOUBLE  COMMENT 'Compounded, never last price over first price',
  annualised_return            DOUBLE,
  income_return                DOUBLE  COMMENT 'Dividends received as a share of the opening price',
  volatility                   DOUBLE  COMMENT 'STDDEV(monthly return) * SQRT(12)',
  index_return_same_period     DOUBLE  COMMENT 'SPY over exactly this span, on the same basis',
  index_volatility_same_period DOUBLE,
  beat_index                   BOOLEAN,
  risk_adjusted_return         DOUBLE  COMMENT 'Annualised return per unit of volatility',
  return_basis                 STRING  COMMENT 'Both sides always measured the same way',
  rank_by_return               INT     COMMENT 'Attribute only. Ranking must never filter this fact',
  rank_by_income               INT,
  rank_by_risk_adjusted        INT
)
COMMENT 'The answer: return, income and risk for every ticker at five horizons';

In [0]:
CREATE OR REPLACE TEMP VIEW gold_stage_horizon AS
WITH clock AS (
  SELECT MAX(month_start) AS as_of_start, MAX(month_key) AS as_of_key
  FROM `index-vs-trust-pipeline`.silver.monthly_performance
),
horizons AS (
  SELECT * FROM VALUES (15), (10), (5), (3), (1) AS h(horizon_years)
),
windows AS (
  SELECT h.horizon_years,
         ADD_MONTHS(c.as_of_start, -(h.horizon_years * 12) + 1) AS win_start,
         c.as_of_start                                          AS win_end,
         c.as_of_key                                            AS as_of_key,
         h.horizon_years * 12                                   AS window_months
  FROM horizons h CROSS JOIN clock c
),
-- SPY compounded once, as a running product in log space. Adding logarithms is the same as
-- multiplying, and SQL has no running product. One pass gives the index value at every
-- month, so its return over any span becomes one point divided by another.
spy_curve AS (
  SELECT month_key,
         EXP(SUM(LN(1 + COALESCE(total_return, 0))) OVER (
             ORDER BY month_key ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)) AS cum_total_incl,
         EXP(SUM(LN(1 + COALESCE(total_return, 0))) OVER (
             ORDER BY month_key ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING)) AS cum_total_excl,
         EXP(SUM(LN(1 + COALESCE(price_return, 0))) OVER (
             ORDER BY month_key ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)) AS cum_price_incl,
         EXP(SUM(LN(1 + COALESCE(price_return, 0))) OVER (
             ORDER BY month_key ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING)) AS cum_price_excl
  FROM `index-vs-trust-pipeline`.silver.monthly_performance
  WHERE ticker = 'SPY'
),
-- Each ticker is measured on the basis it can support. The archive trusts have no dividends,
-- so they are price return only, and the index is then measured the same way.
observations AS (
  SELECT d.ticker_key, d.ticker, w.horizon_years, w.window_months, w.as_of_key,
         CASE WHEN d.price_source = 'archive' THEN 'price' ELSE 'total' END AS return_basis,
         CASE WHEN d.price_source = 'archive' THEN m.price_return ELSE m.total_return END AS r,
         m.month_key, m.close, m.dividend
  FROM `index-vs-trust-pipeline`.gold.dim_ticker d
  CROSS JOIN windows w
  JOIN `index-vs-trust-pipeline`.silver.monthly_performance m
    ON m.ticker = d.ticker AND m.month_start BETWEEN w.win_start AND w.win_end
  WHERE d.is_current
    AND d.months_available >= 36
    AND (CASE WHEN d.price_source = 'archive' THEN m.price_return ELSE m.total_return END) IS NOT NULL
),
aggregated AS (
  SELECT ticker_key, ticker, horizon_years, window_months, as_of_key, return_basis,
         COUNT(*)                AS months_used,
         MIN(month_key)          AS start_month_key,
         MAX(month_key)          AS end_month_key,
         EXP(SUM(LN(1 + r))) - 1 AS total_return,
         STDDEV(r) * SQRT(12)    AS volatility,
         SUM(dividend) / NULLIF(MIN_BY(close, month_key), 0) AS income_return
  FROM observations
  GROUP BY ticker_key, ticker, horizon_years, window_months, as_of_key, return_basis
  -- A ticker must cover at least 80% of the window, or its return is not comparable with one
  -- measured over the whole of it. A delisted trust needs no special case: it is measured
  -- over the months it has, and the index over that same span.
  HAVING COUNT(*) >= 0.8 * window_months
),
compared AS (
  SELECT a.*,
         CASE WHEN a.return_basis = 'price' THEN e.cum_price_incl / s.cum_price_excl - 1
              ELSE e.cum_total_incl / s.cum_total_excl - 1 END AS index_return_same_period
  FROM aggregated a
  JOIN spy_curve s ON s.month_key = a.start_month_key
  JOIN spy_curve e ON e.month_key = a.end_month_key
),
index_risk AS (
  SELECT c.horizon_years, c.start_month_key, c.end_month_key, c.return_basis,
         STDDEV(CASE WHEN c.return_basis = 'price' THEN x.price_return ELSE x.total_return END)
           * SQRT(12) AS index_volatility_same_period
  FROM (SELECT DISTINCT horizon_years, start_month_key, end_month_key, return_basis FROM compared) c
  JOIN `index-vs-trust-pipeline`.silver.monthly_performance x
    ON x.ticker = 'SPY' AND x.month_key BETWEEN c.start_month_key AND c.end_month_key
  GROUP BY c.horizon_years, c.start_month_key, c.end_month_key, c.return_basis
),
final AS (
  SELECT c.*, r.index_volatility_same_period
  FROM compared c
  JOIN index_risk r
    ON r.horizon_years = c.horizon_years
   AND r.start_month_key = c.start_month_key
   AND r.end_month_key = c.end_month_key
   AND r.return_basis = c.return_basis
)
SELECT MD5(CONCAT_WS('|', ticker, CAST(horizon_years AS STRING)))      AS horizon_key,
       ticker_key, ticker, as_of_key AS month_key, horizon_years,
       start_month_key, end_month_key, months_used,
       total_return,
       POWER(1 + total_return, 12.0 / months_used) - 1                 AS annualised_return,
       income_return,
       volatility,
       index_return_same_period,
       index_volatility_same_period,
       total_return > index_return_same_period                         AS beat_index,
       (POWER(1 + total_return, 12.0 / months_used) - 1)
         / NULLIF(volatility, 0)                                       AS risk_adjusted_return,
       return_basis,
       -- Rank is an attribute, never a filter. SPY is ranked alongside the trusts, so its
       -- position is a value you read rather than something the dashboard calculates.
       RANK() OVER (PARTITION BY horizon_years ORDER BY total_return DESC)  AS rank_by_return,
       RANK() OVER (PARTITION BY horizon_years ORDER BY income_return DESC) AS rank_by_income,
       RANK() OVER (PARTITION BY horizon_years
                    ORDER BY (POWER(1 + total_return, 12.0 / months_used) - 1)
                             / NULLIF(volatility, 0) DESC)                  AS rank_by_risk_adjusted
FROM final;

In [0]:
MERGE INTO `index-vs-trust-pipeline`.gold.fact_horizon_performance AS t
USING gold_stage_horizon AS s
   ON t.ticker = s.ticker AND t.horizon_years = s.horizon_years
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

## Verification

In [0]:
SELECT horizon_years,
       COUNT(*)                                                 AS rows,
       SUM(CASE WHEN beat_index THEN 1 ELSE 0 END)              AS beat_count,
       ROUND(100.0 * SUM(CASE WHEN beat_index THEN 1 ELSE 0 END)
             / COUNT(*), 1)                                     AS beat_rate_pct,
       SUM(CASE WHEN months_used < 0.8 * horizon_years * 12
                THEN 1 ELSE 0 END)                              AS under_coverage
FROM `index-vs-trust-pipeline`.gold.fact_horizon_performance
GROUP BY horizon_years
ORDER BY horizon_years DESC;

Expect **445 rows** in total, and `under_coverage` **0** at every horizon.

| horizon | rows | beat rate |
|---|---|---|
| 15 | 79 | around **5%** |
| 10 | 88 | around **11%** |
| 5 | 93 | around **16%** |
| 3 | 93 | around **30%** |
| 1 | 92 | around **42%** |

Each row count includes the 3 index tickers, so the trust counts are 76 / 85 / 90 / 90 / 89.
The beat rate is computed over everything in the table, so it differs slightly from the
trusts-only figure quoted in the presentation — that one filters `entity_type = 'Trust'`.

**If the beat rate is not close to these, the fact does not match the Silver data it was
built from, and the difference is a bug rather than rounding.**

In [0]:
-- The answer, trusts only, exactly as the dashboard will ask for it.
SELECT f.horizon_years,
       COUNT(*)                                                    AS trusts,
       ROUND(100.0 * SUM(CASE WHEN f.beat_index THEN 1 ELSE 0 END)
             / COUNT(*), 1)                                        AS beat_rate_pct,
       ROUND(100 * PERCENTILE_APPROX(f.volatility, 0.5), 1)        AS median_vol_pct,
       ROUND(100 * MAX(f.index_volatility_same_period), 1)         AS spy_vol_pct,
       ROUND(100.0 * SUM(CASE WHEN f.beat_index
                          AND f.volatility < f.index_volatility_same_period
                          THEN 1 ELSE 0 END) / COUNT(*), 1)        AS beat_and_calmer_pct
FROM `index-vs-trust-pipeline`.gold.fact_horizon_performance f
JOIN `index-vs-trust-pipeline`.gold.dim_ticker d
  ON d.ticker_key = f.ticker_key AND d.entity_type = 'Trust'
GROUP BY f.horizon_years
ORDER BY f.horizon_years DESC;

Expect **76 / 85 / 90 / 90 / 89** trusts and beat rates of **5.3 / 10.6 / 15.6 / 30.0 / 41.6**.

The final column is the one to say out loud: **`beat_and_calmer_pct` is 0.0 at both 15 and 10
years.** Not one trust in seventy-six beat the S&P 500 over fifteen years while also being
less volatile than it.

In [0]:
-- Where the index sits on each leaderboard. Rank is stored, so this is a lookup.
SELECT horizon_years, ticker, rank_by_return, rank_by_income, rank_by_risk_adjusted,
       ROUND(100 * total_return, 1) AS total_return_pct,
       ROUND(100 * income_return, 1) AS income_pct
FROM `index-vs-trust-pipeline`.gold.fact_horizon_performance
WHERE ticker = 'SPY'
ORDER BY horizon_years DESC;

Expect **5 rows**, one per horizon. At 10 years SPY should rank around **8th on return** and
around **56th on income** — the single clearest statement of what the index is for.

That contrast is the second finding of the project: *the index wins on growth, the trusts win
on income.*